# Mamba-JEPA 12x12 all-position — unified thesis evaluation

**Evaluation protocol:** `unified_eval_v3`. Run cells in order. The encoder is frozen; all models use the same held-out shards, bf16 encoder execution, fp32 readouts, seeds, training budgets, and final test games.

Each expensive stage is a separate cell and saves immediately under `RUN_DIR/unified_eval/final`, so a Colab disconnect does not erase completed stages.

## 1. Mount Drive, validate runtime/run, build the exact split, and copy `final.pt`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
from importlib import metadata as importlib_metadata
import json
import os
import shutil
import subprocess
import sys
import yaml

PROJECT_ROOT = Path('/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code')
assert (PROJECT_ROOT / 'pyproject.toml').is_file(), PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)

CONFIG_NAME = 'mamba_jepa_v1_hd_vicreg_b12_all_positions.yml'  # The only experiment selector.
CONFIG_PATH = PROJECT_ROOT / 'configs' / CONFIG_NAME
assert CONFIG_PATH.is_file(), CONFIG_PATH
EXPERIMENT_CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
yaml_experiment = EXPERIMENT_CONFIG.get('experiment', {})
yaml_data = EXPERIMENT_CONFIG.get('data', {})
yaml_model = EXPERIMENT_CONFIG.get('model', {})
yaml_objective = EXPERIMENT_CONFIG.get('objective', {})
yaml_paths = EXPERIMENT_CONFIG.get('paths', {})

# The YAML is the single source of truth for experiment identity and paths.
RUN_NAME = str(yaml_experiment.get('name'))
BOARD_SIZE = int(yaml_data.get('board_size'))
ARCHITECTURE = str(yaml_model.get('encoder_architecture', 'transformer'))
OBJECTIVE = str(yaml_objective.get('objective_class', 'jepa'))
DATA_DIR = Path(str(yaml_paths.get('data_dir')))
RUN_DIR = Path(str(yaml_paths.get('drive_sync_dir')))
EXPECTED_POSITION_SAMPLING = str(yaml_objective.get('position_sampling', 'random'))
EXPECTED_VARIANT = str(yaml_experiment.get('variant', 'v1'))
EXPECTED_INTERNAL_VARIANT = f'jepa_{EXPECTED_VARIANT}'
EXPECTED_VIEW_MODE = str(yaml_objective.get('view_mode'))
EXPECTED_LOSS_TYPE = str(yaml_objective.get('loss_type'))
EXPECTED_PREDICTION_HORIZON = int(yaml_objective.get('prediction_horizon', 1))
EXPECTED_USE_EMA = bool(yaml_objective.get('use_ema_target', EXPECTED_VARIANT == 'v2'))

CHECKPOINT_NAME = 'final.pt'  # thesis headline evaluation is always explicit
EXPECTED_PRETRAINING_CHUNKS = 200
MAMBA_SSM_VERSION = '2.3.2.post1'
CAUSAL_CONV1D_VERSION = '1.6.2.post1'
DRIVE_CKPT = RUN_DIR / CHECKPOINT_NAME

# Resume policy for an existing results file in OUTPUT_DIR.
# ALLOW_SOURCE_DRIFT: reuse saved stages when only othello_research/*.py changed.
# OVERWRITE_INCOMPATIBLE: discard the saved stages and recompute everything.
ALLOW_SOURCE_DRIFT = False
OVERWRITE_INCOMPATIBLE = False

assert DATA_DIR.is_dir(), DATA_DIR
assert RUN_DIR.is_dir(), RUN_DIR
assert DRIVE_CKPT.is_file(), (
    f'Expected exact final checkpoint {DRIVE_CKPT}. '
    'No automatic checkpoint fallback is permitted.'
)

def installed_version(distribution_name):
    try:
        return importlib_metadata.version(distribution_name)
    except importlib_metadata.PackageNotFoundError:
        return None

if ARCHITECTURE == 'mamba' and (
    installed_version('mamba-ssm') != MAMBA_SSM_VERSION
    or installed_version('causal-conv1d') != CAUSAL_CONV1D_VERSION
):
    install_env = os.environ.copy()
    install_env['MAMBA_KEEP_CUDA_BUILD'] = 'TRUE'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'packaging', 'ninja'], check=True)
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install',
            f'causal-conv1d=={CAUSAL_CONV1D_VERSION}',
            f'mamba-ssm=={MAMBA_SSM_VERSION}',
            '--no-build-isolation',
        ],
        check=True,
        env=install_env,
    )

import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime in Colab.'
assert torch.cuda.is_bf16_supported(), 'The unified headline protocol requires CUDA bf16 support.'
device = torch.device('cuda:0')
if ARCHITECTURE == 'mamba':
    import mamba_ssm  # noqa: F401 -- fail here instead of using a fallback backend
    assert installed_version('mamba-ssm') == MAMBA_SSM_VERSION
    assert installed_version('causal-conv1d') == CAUSAL_CONV1D_VERSION
    MAMBA_SSM_RUNTIME_VERSION = installed_version('mamba-ssm')
    CAUSAL_CONV1D_RUNTIME_VERSION = installed_version('causal-conv1d')
else:
    MAMBA_SSM_RUNTIME_VERSION = None
    CAUSAL_CONV1D_RUNTIME_VERSION = None

from othello_research.evaluation.unified import (
    UnifiedEvalConfig,
    UnifiedEvaluator,
    build_evaluation_split,
    file_sha256,
)


protocol_config = UnifiedEvalConfig(board_size=BOARD_SIZE)
PROTOCOL_ID = protocol_config.protocol_id
all_chunks = sorted(DATA_DIR.glob('*.pickle'))
split = build_evaluation_split(all_chunks, protocol_config)

# Exact common split: 0:200 / 200:220 / 220:229 / 229:238.
assert split.pretraining == tuple(all_chunks[0:200])
assert split.downstream_train == tuple(all_chunks[200:220])
assert split.selection == tuple(all_chunks[220:229])
assert split.test == tuple(all_chunks[229:238])

DATA_MANIFEST_PATH = DATA_DIR / 'manifest.json'
assert DATA_MANIFEST_PATH.is_file(), (
    f'Authoritative evaluation requires {DATA_MANIFEST_PATH}; corpus counts '
    'must not be inferred by scanning pickle payloads.'
)
DATA_MANIFEST_PAYLOAD = json.loads(DATA_MANIFEST_PATH.read_text(encoding='utf-8'))
SHARD_COUNTS = dict(DATA_MANIFEST_PAYLOAD.get('shard_counts') or ())
for shard in DATA_MANIFEST_PAYLOAD.get('shards') or ():
    name = str(shard['filename'])
    count = int(shard['count'])
    if name in SHARD_COUNTS:
        assert int(SHARD_COUNTS[name]) == count, (name, SHARD_COUNTS[name], count)
    SHARD_COUNTS[name] = count
missing_counts = [path.name for path in all_chunks if path.name not in SHARD_COUNTS]
assert not missing_counts, f'Manifest lacks shard counts: {missing_counts[:3]}'
EXPECTED_PRETRAIN_GAMES = sum(int(SHARD_COUNTS[path.name]) for path in split.pretraining)

SHARD_SHA256 = dict()
for shard in DATA_MANIFEST_PAYLOAD.get('shards') or ():
    if shard.get('sha256'):
        SHARD_SHA256[str(shard['filename'])] = str(shard['sha256'])
if all(path.name in SHARD_SHA256 for path in all_chunks):
    content_digest = hashlib.sha256()
    for path in all_chunks:
        content_digest.update(path.name.encode('utf-8'))
        content_digest.update(SHARD_SHA256[path.name].encode('ascii'))
    SPLIT_CONTENT_SHA256 = content_digest.hexdigest()
    DATA_MANIFEST_IDENTITY = 'per-shard-sha256'
else:
    SPLIT_CONTENT_SHA256 = None
    DATA_MANIFEST_IDENTITY = 'counts-only'

source_digest = hashlib.sha256()
for source_path in sorted((PROJECT_ROOT / 'othello_research').rglob('*.py')):
    relative = source_path.relative_to(PROJECT_ROOT).as_posix().encode('utf-8')
    source_digest.update(len(relative).to_bytes(8, 'big'))
    source_digest.update(relative)
    source_digest.update(source_path.read_bytes())
PROJECT_SOURCE_SHA256 = source_digest.hexdigest()

# Always overwrite the local copy: a stale checkpoint can never survive a rerun.
LOCAL_CKPT_DIR = Path('/content/unified_eval_checkpoints') / RUN_NAME
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CKPT = LOCAL_CKPT_DIR / CHECKPOINT_NAME
shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
CHECKPOINT_SHA256 = file_sha256(LOCAL_CKPT)
CHECKPOINT_BYTES = LOCAL_CKPT.stat().st_size
DATA_MANIFEST_SHA256 = file_sha256(DATA_MANIFEST_PATH)

print('protocol         :', protocol_config.protocol_id)
print('device / bf16    :', torch.cuda.get_device_name(0), torch.cuda.is_bf16_supported())
print('run              :', RUN_NAME)
print('checkpoint       :', DRIVE_CKPT)
print('checkpoint SHA256:', CHECKPOINT_SHA256)
print('checkpoint bytes :', CHECKPOINT_BYTES)
print('data manifest    :', DATA_MANIFEST_PATH)
print('manifest SHA256  :', DATA_MANIFEST_SHA256)
print('manifest identity:', DATA_MANIFEST_IDENTITY)
print('content SHA256   :', SPLIT_CONTENT_SHA256 or 'unavailable (counts-only legacy manifest)')
print('source SHA256    :', PROJECT_SOURCE_SHA256)
print('pretrain games   :', f'{EXPECTED_PRETRAIN_GAMES:,}')
print('split SHA256     :', split.manifest_hash)


## 2. Validate and load the frozen checkpoint; initialize the evaluator

In [ ]:
from othello_research.training.train_jepa import build_model_from_checkpoint

ckpt = torch.load(LOCAL_CKPT, map_location='cpu', weights_only=False)
checkpoint_games_seen = int(ckpt.get('games_seen', -1))
checkpoint_chunk_count = int(ckpt.get('chunk_count', -1))
checkpoint_model_config = dict(ckpt.get('model_config') or ckpt.get('jepa_config') or {})
checkpoint_train_config = dict(ckpt.get('train_config') or {})
assert checkpoint_games_seen == EXPECTED_PRETRAIN_GAMES, (
    f'Checkpoint games_seen={checkpoint_games_seen:,}, but the manifest says '
    f'the exact first-200-shard budget is {EXPECTED_PRETRAIN_GAMES:,}.'
)
assert checkpoint_chunk_count == EXPECTED_PRETRAINING_CHUNKS, (
    checkpoint_chunk_count, EXPECTED_PRETRAINING_CHUNKS
)
assert int(checkpoint_train_config.get('passes_over_data', -1)) == 1
assert checkpoint_model_config.get('objective_class', 'jepa') == 'jepa'
assert checkpoint_model_config.get('view_mode') == EXPECTED_VIEW_MODE
assert checkpoint_model_config.get('loss_type') == EXPECTED_LOSS_TYPE
assert checkpoint_model_config.get('variant') == EXPECTED_INTERNAL_VARIANT
assert int(checkpoint_model_config.get('prediction_horizon')) == EXPECTED_PREDICTION_HORIZON
assert checkpoint_model_config.get('encoder_architecture', 'transformer') == ARCHITECTURE
assert checkpoint_train_config.get('position_sampling', 'random') == EXPECTED_POSITION_SAMPLING
checkpoint_use_ema = checkpoint_model_config.get(
    'use_ema_target',
    checkpoint_model_config.get('variant') == 'jepa_v2',
)
assert bool(checkpoint_use_ema) == EXPECTED_USE_EMA
if ARCHITECTURE == 'mamba':
    assert checkpoint_model_config.get('mamba_backend') == 'mamba_ssm'

jepa_model, model_cfg = build_model_from_checkpoint(ckpt)
jepa_model.load_state_dict(ckpt['model'])
assert int(model_cfg.board_size) == BOARD_SIZE, (model_cfg.board_size, BOARD_SIZE)
assert str(model_cfg.encoder_architecture) == ARCHITECTURE
if ARCHITECTURE == 'mamba':
    assert str(model_cfg.mamba_backend) == 'mamba_ssm'
jepa_model.to(device).eval()
for parameter in jepa_model.parameters():
    parameter.requires_grad_(False)
encoder = jepa_model.context_encoder
model_config_snapshot = dict(vars(model_cfg)) if hasattr(model_cfg, '__dict__') else str(model_cfg)
training_precision = (
    checkpoint_train_config.get('precision')
    or checkpoint_train_config.get('amp_dtype')
    or checkpoint_train_config.get('mixed_precision')
    or ckpt.get('precision')
)
print('checkpoint training precision:', training_precision or 'unreported')
if 'bf16' not in str(training_precision).lower() and 'bfloat16' not in str(training_precision).lower():
    print('INTERPRETATION NOTE: evaluation is unified bf16/fp32, but this checkpoint '
          'was not verified as bf16-trained; retain training precision as a covariate.')

run_metadata = {
    'run_name': RUN_NAME,
    'architecture': ARCHITECTURE,
    'objective': f'{EXPECTED_INTERNAL_VARIANT}_{EXPECTED_LOSS_TYPE}_{EXPECTED_VIEW_MODE}',
    'position_sampling': EXPECTED_POSITION_SAMPLING,
    'board_size': BOARD_SIZE,
    'config_name': CONFIG_NAME,
    'checkpoint_name': CHECKPOINT_NAME,
    'checkpoint_drive_path': str(DRIVE_CKPT),
    'checkpoint_local_path': str(LOCAL_CKPT),
    'checkpoint_sha256': CHECKPOINT_SHA256,
    'checkpoint_bytes': CHECKPOINT_BYTES,
    'data_manifest_path': str(DATA_MANIFEST_PATH) if DATA_MANIFEST_SHA256 else None,
    'data_manifest_sha256': DATA_MANIFEST_SHA256,
    'data_manifest_identity': DATA_MANIFEST_IDENTITY,
    'split_content_sha256': SPLIT_CONTENT_SHA256,
    'project_source_sha256': PROJECT_SOURCE_SHA256,
    'step': ckpt.get('step'),
    'games_seen': ckpt.get('games_seen'),
    'chunk_count': ckpt.get('chunk_count'),
    'expected_pretrain_games': EXPECTED_PRETRAIN_GAMES,
    'training_precision': training_precision,
    'model_config': model_config_snapshot,
    'train_config': checkpoint_train_config,
    'encoder_parameters': sum(p.numel() for p in encoder.parameters()),
    'total_parameters': sum(p.numel() for p in jepa_model.parameters()),
    'mamba_backend': getattr(model_cfg, 'mamba_backend', None),
    'mamba_ssm_version': MAMBA_SSM_RUNTIME_VERSION,
    'causal_conv1d_version': CAUSAL_CONV1D_RUNTIME_VERSION,
    'view_mode': checkpoint_model_config.get('view_mode'),
    'loss_type': checkpoint_model_config.get('loss_type'),
}
OUTPUT_DIR = RUN_DIR / 'unified_eval' / LOCAL_CKPT.stem
evaluator = UnifiedEvaluator(
    encoder=encoder,
    split=split,
    config=protocol_config,
    device=device,
    output_dir=OUTPUT_DIR,
    run_metadata=run_metadata,
    allow_source_drift=ALLOW_SOURCE_DRIFT,
    overwrite_incompatible=OVERWRITE_INCOMPATIBLE,
)
evaluator.print_protocol()
print('artifacts:', OUTPUT_DIR)


## 3. Frozen linear next-move readout

This stage saves its artifact and updates the shared results JSON before returning.

In [ ]:
linear_result = evaluator.run_frozen_head('linear')
linear_result


## 4. Frozen MLP next-move readout

This stage saves its artifact and updates the shared results JSON before returning.

In [ ]:
mlp_result = evaluator.run_frozen_head('mlp')
mlp_result


## 5. Linear board-state probe

This stage saves its artifact and updates the shared results JSON before returning.

In [ ]:
board_linear_result = evaluator.run_board_probe('linear')
board_linear_result


## 6. MLP board-state probe

This stage saves its artifact and updates the shared results JSON before returning.

In [ ]:
board_mlp_result = evaluator.run_board_probe('mlp')
board_mlp_result


## 7. Write and display the consolidated summary

In [ ]:
from IPython.display import Markdown, display
summary_path = evaluator.write_summary()
display(Markdown(Path(summary_path).read_text(encoding='utf-8')))
print('summary:', summary_path)
print('results:', evaluator.results_path)
